<a href="https://colab.research.google.com/github/adityab-tech/Prism/blob/main/W7_Prism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Old repeatative Codes

In [1]:
!git clone https://github.com/shiyu-coder/Kronos.git

Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 371 (delta 66), reused 49 (delta 49), pack-reused 269 (from 1)
Receiving objects: 100% (371/371), 9.30 MiB | 12.17 MiB/s, done.
Resolving deltas: 100% (178/178), done.


In [2]:
!git clone https://github.com/NeoQuasar/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [3]:
%cd Kronos

!pip install -q -r requirements.txt

/content/Kronos
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.4/515.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.8/485.8 kB 23.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.38.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.33.1 which is incompatible.
diffusers 0.38.0 requires safetensors>=0.8.0-rc.0, but you have safetensors 0.6.2 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.33.1 which is incompatible.
tra

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import sys
import yaml
import torch
import pandas as pd

sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")

In [6]:
from model import KronosTokenizer, Kronos
from train_sequential import SequentialTrainer         #puri training process (epochs, loss, optimizer, validation) manage karta hai
from config_loader import CustomFinetuneConfig
from finetune_base_model import create_dataloaders     #csv ko train/val DataLoaders me convert karta hai

In [7]:
dataset_path = "/content/drive/MyDrive/PRISM/processed/RELIANCE.NS_processed.csv"
df = pd.read_csv(dataset_path)
df.head()

,Date,Close,High,Low,Open,Volume,return,log_return,MA5,MA10,MA20,Volatility,Volume_Change,Volume_MA20,market_return,beta,target
0,2019-04-02,615.133118,621.064466,610.883802,618.807021,17526740,-0.001545,-0.001546,607.125757,603.150867,587.337653,0.016157,-0.206535,21556498.85,0.003775,1.569316,-0.010434
1,2019-04-03,608.714844,621.020165,607.298386,616.483114,17169813,-0.010434,-0.010489,607.829541,604.264093,590.638626,0.016430,-0.020365,21548509.20,-0.005912,1.593855,-0.016107
2,2019-04-04,598.910339,612.477168,596.342998,610.396773,18320845,-0.016107,-0.016238,608.165942,603.223895,593.192639,0.017118,0.067038,21685676.45,-0.003946,1.623214,0.000628
3,2019-04-05,599.286560,603.712925,594.461833,602.407152,14717266,0.000628,0.000628,607.625916,602.270007,595.164584,0.016639,-0.196693,21104925.95,0.005859,1.608380,-0.018207
4,2019-04-08,588.375610,600.880142,585.919014,600.216205,19081843,-0.018207,-0.018374,602.084094,601.716705,596.470364,0.017331,0.296562,21172113.50,-0.005267,1.638698,0.003912


In [8]:
df = df.rename(columns={
    "Date":"timestamps",
    "Open":"open",
    "High":"high",
    "Low":"low",
    "Close":"close",
    "Volume":"volume"
})
df["amount"] = 0
df["timestamps"] = pd.to_datetime(df["timestamps"])
df.head()

,timestamps,close,high,low,open,volume,return,log_return,MA5,MA10,MA20,Volatility,Volume_Change,Volume_MA20,market_return,beta,target,amount
0,2019-04-02,615.133118,621.064466,610.883802,618.807021,17526740,-0.001545,-0.001546,607.125757,603.150867,587.337653,0.016157,-0.206535,21556498.85,0.003775,1.569316,-0.010434,0
1,2019-04-03,608.714844,621.020165,607.298386,616.483114,17169813,-0.010434,-0.010489,607.829541,604.264093,590.638626,0.016430,-0.020365,21548509.20,-0.005912,1.593855,-0.016107,0
2,2019-04-04,598.910339,612.477168,596.342998,610.396773,18320845,-0.016107,-0.016238,608.165942,603.223895,593.192639,0.017118,0.067038,21685676.45,-0.003946,1.623214,0.000628,0
3,2019-04-05,599.286560,603.712925,594.461833,602.407152,14717266,0.000628,0.000628,607.625916,602.270007,595.164584,0.016639,-0.196693,21104925.95,0.005859,1.608380,-0.018207,0
4,2019-04-08,588.375610,600.880142,585.919014,600.216205,19081843,-0.018207,-0.018374,602.084094,601.716705,596.470364,0.017331,0.296562,21172113.50,-0.005267,1.638698,0.003912,0


In [9]:
save_path = "/content/Kronos/finetune_csv/data/reliance.csv"
df.to_csv(save_path,index=False)
print(save_path)

/content/Kronos/finetune_csv/data/reliance.csv


In [10]:
config = {
    "data":{
        "data_path":save_path,
        "lookback_window":30,
        "predict_window":5,
        "max_context":30,
        "clip":5,
        "train_ratio":0.8,
        "val_ratio":0.2,
        "test_ratio":0.0
    },

    "training":{
        "tokenizer_epochs":2,
        "basemodel_epochs":2,
        "batch_size":32,
        "learning_rate":1e-4,
        "num_workers":2
    },

    "model_paths":{
        "exp_name":"Reliance",
        "pretrained_tokenizer":"NeoQuasar/Kronos-Tokenizer-base",
        "pretrained_predictor":"NeoQuasar/Kronos-base",
        "base_save_path":"/content/Kronos/checkpoints"
    }
}

In [11]:
config_path="/content/Kronos/finetune_csv/configs/reliance.yaml"
with open(config_path,"w") as f:
    yaml.dump(config,f)
print(config_path)

/content/Kronos/finetune_csv/configs/reliance.yaml


In [12]:
cfg = CustomFinetuneConfig(config_path)         #YAML configuration ko Python object (cfg) me load karo.
print(cfg.data_path)
print(cfg.lookback_window)
print(cfg.batch_size)

/content/Kronos/finetune_csv/data/reliance.csv
30
32


In [13]:
train_loader, val_loader, *_ = create_dataloaders(cfg)      #*_ iska matlab hai baaki sab ignore karo
#DataLoaders banao aur function ke return me se sirf train_loader aur val_loader lo; baaki returned values ko ignore kar do

Creating data loaders...
Original data time range: 2019-04-02 00:00:00 to 2024-12-30 00:00:00
Original data total length: 1417 records
[TRAIN] Training set: first 1133 time points (0.8)
[TRAIN] Training set time range: 2019-04-02 00:00:00 to 2023-11-01 00:00:00
[TRAIN] Data length after split: 1133 records
[TRAIN] Data length: 1133, Available samples: 1098
Original data time range: 2019-04-02 00:00:00 to 2024-12-30 00:00:00
Original data total length: 1417 records
[VAL] Validation set: time points 1134 to 1417 (0.2)
[VAL] Validation set time range: 2023-11-02 00:00:00 to 2024-12-30 00:00:00
[VAL] Data length after split: 284 records
[VAL] Data length: 284, Available samples: 249
Training set size: 1098, Validation set size: 249


In [14]:
batch_x, batch_stamp = next(iter(train_loader))
print(batch_x.shape)
print(batch_stamp.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([32, 36, 6])
torch.Size([32, 36, 5])


#Tokenizer and Outputs

In [15]:
tokenizer = KronosTokenizer.from_pretrained(
    "NeoQuasar/Kronos-Tokenizer-base"
)
model = Kronos.from_pretrained(
    "NeoQuasar/Kronos-base"
)

config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tokenizer.to(device)
model = model.to(device)

In [17]:
trainer = SequentialTrainer(config_path)

Using device: cpu (rank=0, world_size=1, local_rank=0)
Kronos finetuning configuration summary
Experiment name: Reliance
Data path: /content/Kronos/finetune_csv/data/reliance.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [18]:
trainer.train_tokenizer_phase()

2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - === Tokenizer Training Started ===
INFO:tokenizer_training_rank_0:=== Tokenizer Training Started ===
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Experiment Name: Reliance
INFO:tokenizer_training_rank_0:Experiment Name: Reliance
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Log Directory: /content/Kronos/checkpoints/logs
INFO:tokenizer_training_rank_0:Log Directory: /content/Kronos/checkpoints/logs
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Rank: 0
INFO:tokenizer_training_rank_0:Rank: 0
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Timestamp: 2026-07-14 06:09:04
INFO:tokenizer_training_rank_0:Timestamp: 2026-07-14 06:09:04
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Loading pretrained tokenizer...
INFO:tokenizer_training_rank_0:Loading pretrained tokenizer...



Starting Tokenizer Fine-tuning Phase
Tokenizer model exists: False
Basemodel model exists: False
Loading pretrained tokenizer...


2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Tokenizer parameters: 3,958,042
INFO:tokenizer_training_rank_0:Tokenizer parameters: 3,958,042
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - === Training Configuration ===
INFO:tokenizer_training_rank_0:=== Training Configuration ===
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Data path: /content/Kronos/finetune_csv/data/reliance.csv
INFO:tokenizer_training_rank_0:Data path: /content/Kronos/finetune_csv/data/reliance.csv
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Lookback window: 30
INFO:tokenizer_training_rank_0:Lookback window: 30
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Predict window: 5
INFO:tokenizer_training_rank_0:Predict window: 5
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Batch size: 32
INFO:tokenizer_training_rank_0:Batch size: 32
2026-07-14 06:09:04 - tokenizer_training_rank_0 - INFO - Learning rate: 0.0002
INFO:tokenizer_training_rank_0:Learning rate:

Tokenizer parameters: 3,958,042
Starting tokenizer fine-tuning training...
Creating tokenizer training data loaders...
Original data time range: 2019-04-02 00:00:00 to 2024-12-30 00:00:00
Original data total length: 1417 records
[TRAIN] Training set: first 1133 time points (0.8)
[TRAIN] Training set time range: 2019-04-02 00:00:00 to 2023-11-01 00:00:00
[TRAIN] Data length after split: 1133 records
[TRAIN] Data length: 1133, Available samples: 1098
Original data time range: 2019-04-02 00:00:00 to 2024-12-30 00:00:00
Original data total length: 1417 records
[VAL] Validation set: time points 1134 to 1417 (0.2)
[VAL] Validation set time range: 2023-11-02 00:00:00 to 2024-12-30 00:00:00
[VAL] Data length after split: 284 records
[VAL] Data length: 284, Available samples: 249
Training set size: 1098, Validation set size: 249


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
2026-07-14 06:09:51 - tokenizer_training_rank_0 - INFO - 
--- Epoch 1/2 Summary ---
Validation Loss: 0.0224
Epoch Time: 0:00:40
Total Training Time: 0:00:40

INFO:tokenizer_training_rank_0:
--- Epoch 1/2 Summary ---
Validation Loss: 0.0224
Epoch Time: 0:00:40
Total Training Time: 0:00:40

2026-07-14 06:09:51 - tokenizer_training_rank_0 - INFO - Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0224)
INFO:tokenizer_training_rank_0:Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0224)


...

2026-07-14 06:10:13 - tokenizer_training_rank_0 - INFO - [Epoch 2/2, Step 16/34] LR: 0.000031, Loss: -0.0086
INFO:tokenizer_training_rank_0:[Epoch 2/2, Step 16/34] LR: 0.000031, Loss: -0.0086
2026-07-14 06:10:13 - tokenizer_training_rank_0 - INFO -   - VQ Loss: -0.0676
  - Recon Loss Pre: 0.0316
  - Recon Loss All: 0.0188
INFO:tokenizer_training_rank_0:  - VQ Loss: -0.0676
  - Recon Loss Pre: 0.0316
  - Recon Loss All: 0.0188


[Epoch 2/2, Step 16/34] LR: 0.000031, Loss: -0.0086
  - VQ Loss: -0.0676
  - Recon Loss Pre: 0.0316
  - Recon Loss All: 0.0188


2026-07-14 06:10:34 - tokenizer_training_rank_0 - INFO - 
--- Epoch 2/2 Summary ---
Validation Loss: 0.0211
Epoch Time: 0:00:43
Total Training Time: 0:00:43

INFO:tokenizer_training_rank_0:
--- Epoch 2/2 Summary ---
Validation Loss: 0.0211
Epoch Time: 0:00:43
Total Training Time: 0:00:43

2026-07-14 06:10:35 - tokenizer_training_rank_0 - INFO - Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0211)
INFO:tokenizer_training_rank_0:Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0211)
2026-07-14 06:10:35 - tokenizer_training_rank_0 - INFO - Tokenizer training completed! Best validation loss: 0.0211
Training time: 1.51 minutes
Model saved to: /content/Kronos/checkpoints/tokenizer
INFO:tokenizer_training_rank_0:Tokenizer training completed! Best validation loss: 0.0211
Training time: 1.51 minutes
Model saved to: /content/Kronos/checkpoints/tokenizer



--- Epoch 2/2 Summary ---
Validation Loss: 0.0211
Epoch Time: 0:00:43
Total Training Time: 0:00:43

Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0211)

Tokenizer training completed! Best validation loss: 0.0211
Training time: 1.51 minutes
Model saved to: /content/Kronos/checkpoints/tokenizer


True

In [19]:
print(os.path.exists("/content/Kronos/checkpoints/tokenizer"))
print(os.listdir("/content/Kronos/checkpoints"))

True
['logs', 'tokenizer']


In [20]:
!find /content/Kronos/checkpoints -type d            #d for directories while f for files

/content/Kronos/checkpoints
/content/Kronos/checkpoints/logs
/content/Kronos/checkpoints/tokenizer
/content/Kronos/checkpoints/tokenizer/best_model


In [21]:
print("pretrained_tokenizer_path :", cfg.pretrained_tokenizer_path)
print("pretrained_predictor_path :", cfg.pretrained_predictor_path)
print("finetuned_tokenizer_path :", cfg.finetuned_tokenizer_path)
print("base_save_path :", cfg.base_save_path)
print("tokenizer_save_path :", cfg.tokenizer_save_path)
print("basemodel_save_path :", cfg.basemodel_save_path)

pretrained_tokenizer_path : NeoQuasar/Kronos-Tokenizer-base
pretrained_predictor_path : NeoQuasar/Kronos-base
finetuned_tokenizer_path : None
base_save_path : /content/Kronos/checkpoints
tokenizer_save_path : /content/Kronos/checkpoints/tokenizer
basemodel_save_path : /content/Kronos/checkpoints/basemodel


In [22]:
config["model_paths"]["finetuned_tokenizer"] = "/content/Kronos/checkpoints/tokenizer/best_model"
config["model_paths"]["tokenizer_save_name"] = "tokenizer"
config["model_paths"]["basemodel_save_name"] = "basemodel"

In [23]:
with open(config_path, "w") as f:
    yaml.dump(config, f)

In [24]:
cfg = CustomFinetuneConfig(config_path)
print(cfg.finetuned_tokenizer_path)

/content/Kronos/checkpoints/tokenizer/best_model


In [25]:
trainer = SequentialTrainer(config_path)

Using device: cpu (rank=0, world_size=1, local_rank=0)
Kronos finetuning configuration summary
Experiment name: Reliance
Data path: /content/Kronos/finetune_csv/data/reliance.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [26]:
trainer.train_basemodel_phase()

2026-07-14 06:10:35 - basemodel_training_rank_0 - INFO - === Basemodel Training Started ===
INFO:basemodel_training_rank_0:=== Basemodel Training Started ===
2026-07-14 06:10:35 - basemodel_training_rank_0 - INFO - Experiment Name: Reliance
INFO:basemodel_training_rank_0:Experiment Name: Reliance
2026-07-14 06:10:35 - basemodel_training_rank_0 - INFO - Log Directory: /content/Kronos/checkpoints/logs
INFO:basemodel_training_rank_0:Log Directory: /content/Kronos/checkpoints/logs
2026-07-14 06:10:35 - basemodel_training_rank_0 - INFO - Rank: 0
INFO:basemodel_training_rank_0:Rank: 0
2026-07-14 06:10:35 - basemodel_training_rank_0 - INFO - Timestamp: 2026-07-14 06:10:35
INFO:basemodel_training_rank_0:Timestamp: 2026-07-14 06:10:35
2026-07-14 06:10:35 - basemodel_training_rank_0 - INFO - Loading fine-tuned tokenizer...
INFO:basemodel_training_rank_0:Loading fine-tuned tokenizer...



Starting Basemodel Fine-tuning Phase
Tokenizer model exists: True
Basemodel model exists: False
Loading fine-tuned tokenizer...


2026-07-14 06:10:35 - basemodel_training_rank_0 - INFO - Loading pretrained predictor...
INFO:basemodel_training_rank_0:Loading pretrained predictor...


Loading weights from local directory
Loading pretrained predictor...


2026-07-14 06:10:36 - basemodel_training_rank_0 - INFO - Model parameters: 102,310,592
INFO:basemodel_training_rank_0:Model parameters: 102,310,592
2026-07-14 06:10:36 - basemodel_training_rank_0 - INFO - === Training Configuration ===
INFO:basemodel_training_rank_0:=== Training Configuration ===
2026-07-14 06:10:36 - basemodel_training_rank_0 - INFO - Data path: /content/Kronos/finetune_csv/data/reliance.csv
INFO:basemodel_training_rank_0:Data path: /content/Kronos/finetune_csv/data/reliance.csv
2026-07-14 06:10:36 - basemodel_training_rank_0 - INFO - Lookback window: 30
INFO:basemodel_training_rank_0:Lookback window: 30
2026-07-14 06:10:36 - basemodel_training_rank_0 - INFO - Predict window: 5
INFO:basemodel_training_rank_0:Predict window: 5
2026-07-14 06:10:36 - basemodel_training_rank_0 - INFO - Batch size: 32
INFO:basemodel_training_rank_0:Batch size: 32
2026-07-14 06:10:36 - basemodel_training_rank_0 - INFO - Learning rate: 4e-05
INFO:basemodel_training_rank_0:Learning rate: 4e-0

...

2026-07-14 06:26:28 - basemodel_training_rank_0 - INFO - 
--- Epoch 2/2 Summary ---
Training Loss: 3.2514
Validation Loss: 3.5953
Epoch Time: 471.65 seconds

INFO:basemodel_training_rank_0:
--- Epoch 2/2 Summary ---
Training Loss: 3.2514
Validation Loss: 3.5953
Epoch Time: 471.65 seconds




--- Epoch 2/2 Summary ---
Training Loss: 3.2514
Validation Loss: 3.5953
Epoch Time: 471.65 seconds



2026-07-14 06:26:36 - basemodel_training_rank_0 - INFO - Best model saved to: /content/Kronos/checkpoints/basemodel/best_model (validation loss: 3.5953)
INFO:basemodel_training_rank_0:Best model saved to: /content/Kronos/checkpoints/basemodel/best_model (validation loss: 3.5953)
2026-07-14 06:26:36 - basemodel_training_rank_0 - INFO - Basemodel training completed! Best validation loss: 3.5953
Training time: 16.00 minutes
Model saved to: /content/Kronos/checkpoints/basemodel
INFO:basemodel_training_rank_0:Basemodel training completed! Best validation loss: 3.5953
Training time: 16.00 minutes
Model saved to: /content/Kronos/checkpoints/basemodel


Best model saved to: /content/Kronos/checkpoints/basemodel/best_model (validation loss: 3.5953)

Basemodel training completed! Best validation loss: 3.5953
Training time: 16.00 minutes
Model saved to: /content/Kronos/checkpoints/basemodel


True

In [27]:
!find /content/Kronos/checkpoints

/content/Kronos/checkpoints
/content/Kronos/checkpoints/logs
/content/Kronos/checkpoints/logs/basemodel_training_rank_0.log
/content/Kronos/checkpoints/logs/tokenizer_training_rank_0.log
/content/Kronos/checkpoints/basemodel
/content/Kronos/checkpoints/basemodel/best_model
/content/Kronos/checkpoints/basemodel/best_model/model.safetensors
/content/Kronos/checkpoints/basemodel/best_model/README.md
/content/Kronos/checkpoints/basemodel/best_model/config.json
/content/Kronos/checkpoints/tokenizer
/content/Kronos/checkpoints/tokenizer/best_model
/content/Kronos/checkpoints/tokenizer/best_model/model.safetensors
/content/Kronos/checkpoints/tokenizer/best_model/README.md
/content/Kronos/checkpoints/tokenizer/best_model/config.json


In [28]:
print("Training completed successfully.")

Training completed successfully.
